# 过拟合、正则化与早停

## 学习目标

能够根据训练/验证曲线识别过拟合，并区分 Dropout、权重衰减、增强和早停的作用。


## 概念模型与执行路径

正则化不是单一 API：数据增强改变输入分布，权重衰减限制参数规模，Dropout 随机屏蔽激活，早停根据验证集选择训练时刻。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import torch
from common.models import ImageClassifier
plain = ImageClassifier(dropout=0.0)
regularized = ImageClassifier(dropout=0.5)
print(regularized.classifier)


### 实验 3


In [ ]:
sample = torch.randn(4, 1, 28, 28)
regularized.train()
first = regularized(sample)
second = regularized(sample)
regularized.eval()
third = regularized(sample)
fourth = regularized(sample)
print("train outputs equal:", torch.allclose(first, second))
print("eval outputs equal:", torch.allclose(third, fourth))


### 实验 4


In [ ]:
optimizer = torch.optim.AdamW(regularized.parameters(), lr=1e-3, weight_decay=1e-4)
print("weight decay:", optimizer.param_groups[0]["weight_decay"])


### 实验 5


In [ ]:
import matplotlib.pyplot as plt
train_loss = [1.0, .7, .45, .25, .12, .06]
validation_loss = [1.1, .75, .5, .42, .55, .8]
plt.plot(train_loss, label="train")
plt.plot(validation_loss, label="validation")
plt.axvline(3, color="black", linestyle="--", label="best checkpoint")
plt.legend(); plt.show()


## 底层机制

Dropout 只在 train 模式随机屏蔽激活。AdamW 的 decoupled weight decay 与把 L2 项直接混入 Adam 梯度并不完全等价。早停 patience 应允许短期噪声。


## 检查点

当训练损失下降而验证损失连续上升时，这是优化失败还是泛化失败？应保存哪一轮？


## 试一试

分别关闭增强、Dropout 和 weight decay，记录 quick 训练的训练/验证差距，而不是只比较训练准确率。


## 常见错误与调试

用测试集调早停、忘记 eval 导致 Dropout 波动、正则化过强导致欠拟合、只看 accuracy 忽略 loss 趋势。
